# Model test & validation: k-Nearest Neighbors (k=3, distance-weighted)

A 'grid memorization' diagnostic per the project context, not a serious final model. Key edge case: distance-weighted kNN gives near-infinite weight to an exact or near-exact match -- verify this behaves sanely rather than dividing by zero, and confirm kNN cannot extrapolate (its neighbors are always drawn from the training range, same failure mode as trees).

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
warnings.filterwarnings("ignore")

def mape(y_true, y_pred):
    return float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)
def r2(y_true, y_pred):
    return float(r2_score(y_true, y_pred))

df_raw = pd.read_csv("../../data/chf_long_clean.csv")
df = df_raw[df_raw.X != 1.0].reset_index(drop=True)
FEATURES = ["P", "G", "X"]
TARGET = "CHF"
sorted_P = sorted(df.P.unique())

# Split A (random, seed 0) -- quick interpolation check
X_all, y_all = df[FEATURES].values, df[TARGET].values
XtrA, XteA, ytrA, yteA = train_test_split(X_all, y_all, test_size=0.2, random_state=0)

# Split C (edge extrapolation) -- the honest test
train_dfC = df[df.P <= 16000].reset_index(drop=True)
test_dfC = df[df.P >= 17000].reset_index(drop=True)
XtrC, ytrC = train_dfC[FEATURES].values, train_dfC[TARGET].values
XteC, yteC = test_dfC[FEATURES].values, test_dfC[TARGET].values

print(f"Split A: {len(XtrA)} train / {len(XteA)} test")
print(f"Split C: {len(XtrC)} train / {len(XteC)} test")


## Fit on Split A (interpolation) and Split C (extrapolation)

In [ ]:

scaler = StandardScaler().fit(XtrA)
model_A = KNeighborsRegressor(n_neighbors=3, weights="distance").fit(scaler.transform(XtrA), np.log(ytrA))
predA = np.exp(model_A.predict(scaler.transform(XteA)))
print(f"Split A: R2={r2(yteA, predA):.4f}, MAPE={mape(yteA, predA):.2f}%")

scalerC = StandardScaler().fit(XtrC)
model_C = KNeighborsRegressor(n_neighbors=3, weights="distance").fit(scalerC.transform(XtrC), np.log(ytrC))
predC = np.exp(model_C.predict(scalerC.transform(XteC)))
print(f"Split C: R2={r2(yteC, predC):.4f}, MAPE={mape(yteC, predC):.2f}%")


## Edge-case tests

(1) Query EXACTLY at a training point -- distance-weighted kNN with an exact match should return that point's value with no numerical warning/error. (2) Query far beyond the training pressure range and confirm the prediction plateaus (all 3 nearest neighbors are pinned at the training boundary, so the prediction cannot continue any trend -- the same structural limitation as trees).

In [ ]:

# (1) exact-match query
exact_point = XtrA[0:1]
exact_scaled = scaler.transform(exact_point)
with np.errstate(divide="raise"):
    pred_exact = np.exp(model_A.predict(exact_scaled))
print(f"Exact-match query: predicted={pred_exact[0]:.2f}, true={ytrA[0]:.2f} -- "
      f"{'PASS (near-exact recovery)' if abs(pred_exact[0]-ytrA[0]) < 1.0 else 'FAIL'}")

# (2) far-extrapolation plateau check
G_fixed, X_fixed = 2000.0, 0.0
p_probe = np.array([16000, 17000, 19000, 25000, 50000, 100000], dtype=float)
probe_pts = np.column_stack([p_probe, np.full_like(p_probe, G_fixed), np.full_like(p_probe, X_fixed)])
probe_pred = np.exp(model_C.predict(scalerC.transform(probe_pts)))
print("\nPrediction vs. probe pressure (G=2000, X=0.0), training P maxes out at 16000:")
for p, pred in zip(p_probe, probe_pred):
    print(f"  P={p:>7.0f} kPa -> predicted CHF={pred:.1f}")
print(f"\nPredictions at P=25000, 50000, 100000 are identical: "
      f"{'PASS' if np.allclose(probe_pred[-3:], probe_pred[-1]) else 'FAIL'} "
      f"(kNN cannot extrapolate past its 3 nearest -- fixed -- training neighbors)")


## Diagnostic plot

In [ ]:

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(p_probe, probe_pred, "o-")
ax.axvline(16000, color="gray", linestyle="--", label="max training P")
ax.set_xscale("log")
ax.set_xlabel("Query pressure (kPa, log scale)")
ax.set_ylabel("Predicted CHF")
ax.set_title("kNN prediction plateaus beyond the training pressure range")
ax.legend()
plt.tight_layout()
plt.savefig("../results/model_tests_knn.png", dpi=100)
plt.show()
